# 🚆 Train Ridership Forecasting — LSTM Model

This notebook builds an **LSTM (Long Short-Term Memory)** model to forecast train ridership and allocate the required number of trains per station.

### Pipeline Overview
1. Load & clean data (drop low-frequency stations)
2. Feature engineering (calendar, lag, rolling features)
3. Time-based train/test split
4. Scale features with MinMaxScaler
5. Build per-station sequences for LSTM input
6. Train LSTM model with early stopping
7. Evaluate and compare with ElasticNet baseline
8. Allocate trains based on predicted ridership

## 1. Setup & Imports

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings

from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_squared_error, r2_score

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout, BatchNormalization
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau

warnings.filterwarnings('ignore')
np.random.seed(42)

ValueError: numpy.dtype size changed, may indicate binary incompatibility. Expected 96 from C header, got 88 from PyObject

## 2. Load Data

In [ ]:
df = pd.read_csv('../data/Ridership.csv')

# Parse date column from Year, Month, Day
df['date'] = pd.to_datetime(
    df['Year'].astype(str) + '-' + df['Month'] + '-' + df['Day'].astype(str)
)

print(f'Dataset shape: {df.shape}')
df.head()

## 3. Filter Low-Frequency Stations

Stations with fewer than 30 records are dropped entirely.

> **Why?** LSTM relies on temporal sequences — merging rare stations into an `Other` group mixes unrelated time series and confuses the model. Since our goal is per-station train allocation, stations without enough data are not reliably forecastable anyway.

In [ ]:
MIN_RECORDS = 30

station_counts = df['Station'].value_counts()
valid_stations = station_counts[station_counts >= MIN_RECORDS].index

dropped = station_counts[station_counts < MIN_RECORDS]
print(f'Dropped {len(dropped)} rare station(s): {list(dropped.index)}')
print(f'Keeping {len(valid_stations)} station(s)')

df = df[df['Station'].isin(valid_stations)].reset_index(drop=True)
df = df.sort_values('date').reset_index(drop=True)

print(f'\nFiltered dataset shape: {df.shape}')

## 4. Feature Engineering Functions

These are the same helper functions from the ElasticNet notebook — fully reusable here.

In [ ]:
def time_split_by_station(df, split_date):
    """
    Split data into train/test sets using a time boundary.
    The split is performed per station to preserve temporal order within each group.
    """
    train_list, test_list = [], []
    for station, g in df.groupby('Station'):
        g = g.sort_values('date')
        train_list.append(g[g['date'] < split_date])
        test_list.append(g[g['date'] >= split_date])
    return pd.concat(train_list), pd.concat(test_list)


def add_calendar_features(df):
    """
    Add cyclical month encoding and weekend flag.
    Cyclical encoding preserves the circular nature of months (Dec → Jan continuity).
    """
    df = df.copy()
    df['month'] = df['date'].dt.month
    df['month_sin'] = np.sin(2 * np.pi * df['month'] / 12)
    df['month_cos'] = np.cos(2 * np.pi * df['month'] / 12)
    df['day_of_week'] = df['date'].dt.dayofweek
    df['is_weekend'] = df['day_of_week'].isin([5, 6]).astype(int)
    return df


def create_time_series_features(df):
    """
    Create lag and rolling mean features grouped by Station + Period.
    All features are shift(1)-based to prevent data leakage.
    """
    df = df.copy()
    df = df.sort_values(by=['Station', 'date'])

    for lag in [1, 2, 3]:
        df[f'lag_{lag}_period'] = (
            df.groupby(['Station', 'Period'])['Ridership'].shift(lag)
        )

    df['rolling_mean_3_period'] = (
        df.groupby(['Station', 'Period'])['Ridership']
          .transform(lambda x: x.shift(1).rolling(3, min_periods=1).mean())
    )

    df['rolling_mean_7_period'] = (
        df.groupby(['Station', 'Period'])['Ridership']
          .transform(lambda x: x.shift(1).rolling(7, min_periods=1).mean())
    )

    return df


def target_encode_station(train_df, test_df, target_col='Ridership'):
    """
    Time-safe target encoding for the Station column.
    Encoding is fitted on train only and applied to test to avoid leakage.
    Unseen stations in test are filled with the global mean.
    """
    station_mean = train_df.groupby('Station')[target_col].mean()
    global_mean  = train_df[target_col].mean()

    train_df = train_df.copy()
    test_df  = test_df.copy()

    train_df['Station_enc'] = train_df['Station'].map(station_mean)
    test_df['Station_enc']  = test_df['Station'].map(station_mean).fillna(global_mean)

    return train_df, test_df


def allocate_trains(df, capacity=600, min_trains=1):
    """
    Calculate the number of trains needed based on predicted ridership.
    Uses ceiling division so no passenger is left behind.
    """
    df = df.copy()
    df['required_trains'] = np.ceil(
        df['predicted_ridership'] / capacity
    ).astype(int).clip(lower=min_trains)
    return df

## 5. Preprocessing Pipeline

In [ ]:
# --- Calendar features ---
df = add_calendar_features(df)

# --- Time-based split (per station to preserve order) ---
SPLIT_DATE = '2022-01-01'
train_df, test_df = time_split_by_station(df, split_date=SPLIT_DATE)
print(f'Train size: {len(train_df):,} | Test size: {len(test_df):,}')

# --- Outlier removal using TRAIN statistics only ---
Q1 = train_df['Ridership'].quantile(0.25)
Q3 = train_df['Ridership'].quantile(0.75)
IQR = Q3 - Q1
lower_bound  = Q1 - 1.5 * IQR
upper_bound  = Q3 + 1.5 * IQR
median_value = train_df['Ridership'].median()

for split in [train_df, test_df]:
    mask = (split['Ridership'] < lower_bound) | (split['Ridership'] > upper_bound)
    split.loc[mask, 'Ridership'] = median_value

# --- Lag & rolling features (created AFTER split to prevent leakage) ---
train_df = create_time_series_features(train_df)
test_df  = create_time_series_features(test_df)

train_df = train_df.dropna().reset_index(drop=True)
test_df  = test_df.dropna().reset_index(drop=True)

# --- Target encoding for Station ---
train_df, test_df = target_encode_station(train_df, test_df)

print(f'\nAfter dropna — Train: {len(train_df):,} | Test: {len(test_df):,}')

## 6. Encode Categorical Features & Scale

LSTM requires all inputs to be numerical and scaled to [0, 1].
We use `pd.get_dummies` for categoricals and `MinMaxScaler` for all features.

In [ ]:
NUMERIC_FEATURES = [
    'Station_enc', 'Covid19',
    'month_sin', 'month_cos', 'day_of_week', 'is_weekend',
    'lag_1_period', 'lag_2_period', 'lag_3_period',
    'rolling_mean_3_period', 'rolling_mean_7_period'
]

CATEGORICAL_FEATURES = ['Corridor', 'Workday', 'Period']

# One-hot encode categoricals (fit on train, align test columns)
cat_train = pd.get_dummies(train_df[CATEGORICAL_FEATURES])
cat_test  = pd.get_dummies(test_df[CATEGORICAL_FEATURES])
cat_test  = cat_test.reindex(columns=cat_train.columns, fill_value=0)

X_train_raw = pd.concat([train_df[NUMERIC_FEATURES], cat_train], axis=1)
X_test_raw  = pd.concat([test_df[NUMERIC_FEATURES],  cat_test],  axis=1)

# Scale features to [0, 1] — fit only on train
scaler_X = MinMaxScaler()
scaler_y = MinMaxScaler()

X_train_scaled = scaler_X.fit_transform(X_train_raw)
X_test_scaled  = scaler_X.transform(X_test_raw)

y_train_scaled = scaler_y.fit_transform(train_df[['Ridership']])
y_test_scaled  = scaler_y.transform(test_df[['Ridership']])

print(f'Feature matrix shape: {X_train_scaled.shape}')
print(f'Number of features: {X_train_scaled.shape[1]}')

## 7. Build Sequences Per Station

LSTM expects input of shape `(samples, timesteps, features)`.

**Critical:** sequences must be built **per station** — never across stations.
If we built sequences across the full DataFrame, the last row of Station A
would be followed by the first row of Station B, creating a meaningless transition.

In [ ]:
def create_sequences_per_station(X_scaled, y_scaled, station_array, window=7):
    """
    Build sliding-window sequences grouped by station.

    Parameters
    ----------
    X_scaled      : np.ndarray, scaled feature matrix
    y_scaled      : np.ndarray, scaled target column
    station_array : np.ndarray, station label for each row
    window        : int, number of past timesteps to include in each sequence

    Returns
    -------
    Xs      : np.ndarray, shape (n_samples, window, n_features)
    ys      : np.ndarray, shape (n_samples, 1)
    indices : np.ndarray, original row index in the input DataFrame for each sample
    """
    Xs, ys, indices = [], [], []

    for station in np.unique(station_array):
        # Get original row indices for this station
        mask = np.where(station_array == station)[0]
        X_s  = X_scaled[mask]
        y_s  = y_scaled[mask]

        for i in range(window, len(X_s)):
            Xs.append(X_s[i - window:i])  # past `window` steps as input
            ys.append(y_s[i])             # current step as target
            indices.append(mask[i])       # track original DataFrame row index

    return np.array(Xs), np.array(ys), np.array(indices)


WINDOW = 7  # use the past 7 timesteps to predict the next one

X_train_seq, y_train_seq, train_indices = create_sequences_per_station(
    X_train_scaled, y_train_scaled, train_df['Station'].values, window=WINDOW
)
X_test_seq, y_test_seq, test_indices = create_sequences_per_station(
    X_test_scaled, y_test_scaled, test_df['Station'].values, window=WINDOW
)

print(f'Train sequences : {X_train_seq.shape}  → (samples, timesteps, features)')
print(f'Test  sequences : {X_test_seq.shape}')

## 8. Build LSTM Model

Architecture:
- Two stacked LSTM layers to capture short and long-range patterns
- Dropout after each LSTM layer to reduce overfitting
- BatchNormalization to stabilize training
- Dense output layer for regression

In [ ]:
def build_lstm(input_shape):
    """
    Build a stacked LSTM regression model.

    Parameters
    ----------
    input_shape : tuple, (timesteps, n_features)
    """
    model = Sequential([
        LSTM(64, return_sequences=True, input_shape=input_shape),
        Dropout(0.2),
        LSTM(32, return_sequences=False),
        Dropout(0.2),
        BatchNormalization(),
        Dense(16, activation='relu'),
        Dense(1)  # single output: predicted ridership
    ])
    model.compile(optimizer='adam', loss='mse', metrics=['mae'])
    return model


model_lstm = build_lstm(input_shape=(WINDOW, X_train_seq.shape[2]))
model_lstm.summary()

## 9. Train the Model

In [ ]:
callbacks = [
    # Stop training when val_loss stops improving
    EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True, verbose=1),
    # Reduce learning rate when training stalls
    ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=5, verbose=1)
]

history = model_lstm.fit(
    X_train_seq, y_train_seq,
    validation_split=0.1,
    epochs=100,
    batch_size=32,
    callbacks=callbacks,
    verbose=1
)

## 10. Training History

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

# Loss curve
axes[0].plot(history.history['loss'],     label='Train Loss')
axes[0].plot(history.history['val_loss'], label='Val Loss')
axes[0].set_title('MSE Loss')
axes[0].set_xlabel('Epoch')
axes[0].legend()
axes[0].grid(True)

# MAE curve
axes[1].plot(history.history['mae'],     label='Train MAE')
axes[1].plot(history.history['val_mae'], label='Val MAE')
axes[1].set_title('Mean Absolute Error')
axes[1].set_xlabel('Epoch')
axes[1].legend()
axes[1].grid(True)

plt.tight_layout()
plt.show()

## 11. Evaluate on Test Set

In [ ]:
# Predict and inverse-transform back to original ridership scale
y_pred_scaled = model_lstm.predict(X_test_seq)
y_pred = scaler_y.inverse_transform(y_pred_scaled)
y_true = scaler_y.inverse_transform(y_test_seq)

rmse = np.sqrt(mean_squared_error(y_true, y_pred))
r2   = r2_score(y_true, y_pred)

print('=' * 40)
print(f'  LSTM Model Performance')
print('=' * 40)
print(f'  RMSE : {rmse:.2f}')
print(f'  R²   : {r2:.4f}')
print('=' * 40)

## 12. Actual vs Predicted — Visual Check

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# --- Time series plot (first 300 samples) ---
n_show = 300
axes[0].plot(y_true[:n_show],  label='Actual',    alpha=0.8)
axes[0].plot(y_pred[:n_show],  label='Predicted', alpha=0.8, linestyle='--')
axes[0].set_title(f'Actual vs Predicted Ridership (first {n_show} samples)')
axes[0].set_xlabel('Sample index')
axes[0].set_ylabel('Ridership')
axes[0].legend()
axes[0].grid(True)

# --- Scatter plot ---
axes[1].scatter(y_true, y_pred, alpha=0.3, s=10)
lims = [min(y_true.min(), y_pred.min()), max(y_true.max(), y_pred.max())]
axes[1].plot(lims, lims, 'r--', label='Perfect prediction')
axes[1].set_title(f'Scatter Plot  (R² = {r2:.3f})')
axes[1].set_xlabel('Actual Ridership')
axes[1].set_ylabel('Predicted Ridership')
axes[1].legend()
axes[1].grid(True)

plt.tight_layout()
plt.show()

## 13. Per-Station Performance

In [ ]:
# Attach predictions back to the correct test_df rows using tracked indices
results_df = test_df.iloc[test_indices].copy().reset_index(drop=True)
results_df['predicted_ridership'] = y_pred.flatten()
results_df['actual_ridership']    = y_true.flatten()

# Compute RMSE and R2 per station
station_metrics = []
for station, grp in results_df.groupby('Station'):
    s_rmse = np.sqrt(mean_squared_error(grp['actual_ridership'], grp['predicted_ridership']))
    s_r2   = r2_score(grp['actual_ridership'], grp['predicted_ridership'])
    station_metrics.append({'Station': station, 'RMSE': round(s_rmse, 2), 'R2': round(s_r2, 4)})

metrics_df = pd.DataFrame(station_metrics).sort_values('RMSE')
print(metrics_df.to_string(index=False))

# Bar chart
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

axes[0].bar(metrics_df['Station'], metrics_df['RMSE'], color='steelblue')
axes[0].set_title('RMSE per Station')
axes[0].set_xlabel('Station')
axes[0].set_ylabel('RMSE')
axes[0].tick_params(axis='x', rotation=45)
axes[0].grid(axis='y')

axes[1].bar(metrics_df['Station'], metrics_df['R2'], color='darkorange')
axes[1].set_title('R² per Station')
axes[1].set_xlabel('Station')
axes[1].set_ylabel('R²')
axes[1].tick_params(axis='x', rotation=45)
axes[1].grid(axis='y')

plt.tight_layout()
plt.show()

## 14. Train Allocation

Predict the number of trains needed per station using ceiling division:

$$\text{required\_trains} = \lceil \frac{\text{predicted\_ridership}}{\text{capacity}} \rceil$$

In [ ]:
TRAIN_CAPACITY = 600  # passengers per train

results_df = allocate_trains(results_df, capacity=TRAIN_CAPACITY, min_trains=1)

# Summarise by station, date and period
allocation_summary = (
    results_df
    .groupby(['Station', 'date', 'Period'], as_index=False)
    .agg(
        predicted_ridership=('predicted_ridership', 'sum'),
        required_trains=('required_trains', 'max')
    )
)

print(f'Allocation summary shape: {allocation_summary.shape}')
allocation_summary.head(10)

## 15. Allocation Heatmap — Trains per Station & Period

In [ ]:
pivot = allocation_summary.pivot_table(
    index='Station', columns='Period', values='required_trains', aggfunc='mean'
)

plt.figure(figsize=(10, 5))
sns.heatmap(pivot, annot=True, fmt='.1f', cmap='YlOrRd', linewidths=0.5)
plt.title('Average Required Trains — Station × Period')
plt.tight_layout()
plt.show()

## 16. Save Model & Outputs

In [ ]:
import joblib

# Save the Keras model
model_lstm.save('lstm_model.h5')

# Save scalers so they can be reused for inference on new data
joblib.dump(scaler_X, 'scaler_X.pkl')
joblib.dump(scaler_y, 'scaler_y.pkl')

# Save allocation summary to CSV
allocation_summary.to_csv('lstm_allocation_summary.csv', index=False)

print('Saved: lstm_model.h5 | scaler_X.pkl | scaler_y.pkl | lstm_allocation_summary.csv')

## 17. Load Model & Run Inference on New Data

Use this cell to load the saved model and predict on any new incoming data.

In [ ]:
from tensorflow.keras.models import load_model
from tensorflow.keras.metrics import MeanSquaredError

# Load everything
loaded_model = load_model(
    'lstm_model.h5',
    custom_objects={'mse': MeanSquaredError()}
)
loaded_scaler_X = joblib.load('../models/LSTM_model/scaler_X.pkl')
loaded_scaler_y = joblib.load('../models/LSTM_model/scaler_y.pkl')

# --- Example: predict on new_df (must have the same columns as the training data) ---
# new_df = pd.read_csv('../data/new_ridership.csv')
# new_df['date'] = pd.to_datetime(...)
# new_df = add_calendar_features(new_df)
# new_df = create_time_series_features(new_df).dropna()
#
# cat_new = pd.get_dummies(new_df[CATEGORICAL_FEATURES]).reindex(columns=cat_train.columns, fill_value=0)
# X_new   = pd.concat([new_df[NUMERIC_FEATURES], cat_new], axis=1)
# X_new_scaled = loaded_scaler_X.transform(X_new)
#
# X_new_seq, _, new_indices = create_sequences_per_station(
#     X_new_scaled, np.zeros((len(X_new_scaled), 1)), new_df['Station'].values, window=WINDOW
# )
# preds = loaded_scaler_y.inverse_transform(loaded_model.predict(X_new_seq))

print('Model loaded successfully. Uncomment the code above to run inference on new data.')